In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
cat_dog = pd.read_csv('cat_dog_classifier.csv')

In [ ]:
cat_dog.info()

In [ ]:
X = cat_dog.drop('Animal', axis=1)
y = cat_dog['Animal']

In [ ]:
np.random.seed(42)
train_size = 0.8

idx_0 = np.where(y == 0)[0]
idx_1 = np.where(y == 1)[0]

np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

train_0 = int(len(idx_0) * train_size)
train_1 = int(len(idx_1) * train_size)

train_idx = np.concatenate([idx_0[:train_0], idx_1[:train_1]])
test_idx = np.concatenate([idx_0[train_0:], idx_1[train_1:]])

np.random.shuffle(train_idx)
np.random.shuffle(test_idx)

X_train = X[train_idx]
y_train = y[train_idx]
X_test = X[test_idx]
y_test = y[test_idx]

In [ ]:
def knn(X_train, y_train, X_test, k=3):
    distance = np.sqrt(np.sum((X_train - X_test)**2, axis=1))
    sort = np.argsort(distance)
    nearest = sort[:k]
    nearest_label = y_train[nearest]

    label, count = np.unique(nearest_label.astype(int), return_counts=True)
    prediction = label[count.argmax()]

    return prediction

In [ ]:
accuaracies = []
K_range = range(2,11)

for K in K_range:
    correct = 0
    for i in range(len(X_test)):
        pred = knn(X_train, y_train, X_test[i], k=K)
        if pred == y_test[i]:
            correct += 1
    acc = correct / len(X_test)
    accuaracies.append(acc)
    print(f'K   : {K}, accurasies   : {acc}')

best_k = K_range[np.argmax(accuaracies)]
print(f'Best K  : {best_k},accuracies   : {max(accuaracies)}')

In [ ]:
y_pred = []
train_pred = []

for i in range(len(X_test)):
    test = knn(X_train, y_train, X_test[i], k=best_k)
    train = knn(X_train, y_train, X_test[i], k=best_k)
    
    y_pred.append(test)
    train_pred.append(train)

In [ ]:
def evaluate(y_true, y_pred):
    tp = np.sum((y_true ==1) & (y_pred == 1))
    tn = np.sum((y_true ==0) & (y_pred == 0))
    fp = np.sum((y_true ==0) & (y_pred == 1))
    fn = np.sum((y_true ==1) & (y_pred == 0))

    accuracy = (tp + tn ) / (tp + tn + fp + fn)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    recall = tp / (tp + fp) if (tp + fp) > 0 else 0